In [ ]:
using Revise
using Pkg
Pkg.activate("..")
# Pkg.resolve()
# Pkg.instantiate()
using GeneralizedPerturbedEquilibrium, Plots
gr() 

# Read CHEASE output

## Read efit

In [ ]:
println("Loading EFIT data...")
efit_control = GeneralizedPerturbedEquilibrium.Equilibrium.EquilibriumControl(;
    eq_filename="CHEASE_INP1/EQDSK_COCOS_02",
    eq_type="efit",
    jac_type="boozer",
    grid_type="ldp",
    psilow=0.01,
    psihigh=0.994
)
efit_config = GeneralizedPerturbedEquilibrium.Equilibrium.EquilibriumConfig(efit_control, GeneralizedPerturbedEquilibrium.Equilibrium.EquilibriumOutput())
plasma_eq_efit = GeneralizedPerturbedEquilibrium.Equilibrium.setup_equilibrium(efit_config)

## Read INP1_UNFORMATTED (BINARY)

use eq_type as chease(legacy from GPEC) or chease_binary

In [ ]:
println("Loading CHEASE INP1 data...")
chease_control = GeneralizedPerturbedEquilibrium.Equilibrium.EquilibriumControl(;
    eq_filename="CHEASE_INP1/INP1_binary",
    eq_type="chease_binary",
    jac_type="boozer",
    grid_type="ldp",
    psilow=0.01,
    psihigh=0.994,
    r0exp=6.8,
    b0exp=7.4
)
chease_config_chease_binary = GeneralizedPerturbedEquilibrium.Equilibrium.EquilibriumConfig(chease_control, GeneralizedPerturbedEquilibrium.Equilibrium.EquilibriumOutput())
plasma_eq_chease_binary = GeneralizedPerturbedEquilibrium.Equilibrium.setup_equilibrium(chease_config_chease_binary)

## Read INP1_FORMATTED (ASCII)

In [ ]:
println("Loading CHEASE INP1 data...")
chease_control = GeneralizedPerturbedEquilibrium.Equilibrium.EquilibriumControl(;
    eq_filename="CHEASE_INP1/INP1_ascii",
    eq_type="chease_ascii",
    jac_type="boozer",
    grid_type="ldp",
    psilow=0.01,
    psihigh=0.994,
    r0exp=6.8,
    b0exp=7.4
)
chease_config_chease_ascii = GeneralizedPerturbedEquilibrium.Equilibrium.EquilibriumConfig(chease_control, GeneralizedPerturbedEquilibrium.Equilibrium.EquilibriumOutput())
plasma_eq_chease_ascii = GeneralizedPerturbedEquilibrium.Equilibrium.setup_equilibrium(chease_config_chease_ascii)

In [ ]:
# Common evaluation grid for comparison
psi_eval = collect(range(0.0, 1.0, length=200));

# Comparison

## 1-D profile

In [ ]:
println("\n--- Comparing 1D Profiles (EFIT vs CHEASE ASCII/BINARY) ---")

# Evaluate splines
prof_efit = GeneralizedPerturbedEquilibrium.Spl.spline_eval(plasma_eq_efit.sq, psi_eval, 0)
prof_chease_binary = GeneralizedPerturbedEquilibrium.Spl.spline_eval(plasma_eq_chease_binary.sq, psi_eval, 0)
prof_chease_ascii  = GeneralizedPerturbedEquilibrium.Spl.spline_eval(plasma_eq_chease_ascii.sq, psi_eval, 0)

lw_val = 2

# 1. Plot Safety Factor (q)
p1 = plot(psi_eval, prof_efit[:, 4], label="EFIT (G-file)", color=:blue, lw=lw_val)
plot!(p1, psi_eval, prof_chease_ascii[:, 4], label="CHEASE (ASCII)", color=:red, ls=:dash, lw=lw_val)
plot!(p1, psi_eval, prof_chease_binary[:, 4], label="CHEASE (Binary)", color=:green, ls=:dot, lw=lw_val)
title!(p1, "Safety Factor (q) Comparison")
xlabel!(p1, "Normalized Psi (ψₙ)")

# 2. Plot Pressure (P*μ₀)
p2 = plot(psi_eval, prof_efit[:, 2], label="EFIT", color=:blue, lw=lw_val)
plot!(p2, psi_eval, prof_chease_ascii[:, 2], label="CHEASE (ASCII)", color=:red, ls=:dash, lw=lw_val)
plot!(p2, psi_eval, prof_chease_binary[:, 2], label="CHEASE (Binary)", color=:green, ls=:dot, lw=lw_val)
title!(p2, "Pressure (P) Comparison")
xlabel!(p2, "Normalized Psi (ψₙ)")

# 3. Plot Toroidal Field Function (F)
p3 = plot(psi_eval, prof_efit[:, 1], label="EFIT", color=:blue, lw=lw_val)
plot!(p3, psi_eval, prof_chease_ascii[:, 1], label="CHEASE (ASCII)", color=:red, ls=:dash, lw=lw_val)
plot!(p3, psi_eval, prof_chease_binary[:, 1], label="CHEASE (Binary)", color=:green, ls=:dot, lw=lw_val)
title!(p3, "Toroidal Field Fn (F) Comparison")
xlabel!(p3, "Normalized Psi (ψₙ)")

# Layout profiles in a grid
p_profiles = plot(p1, p2, p3, layout=(1, 3), size=(1300, 450))
display(p_profiles)

## Flux surfaces

In [ ]:
println("\n--- Comparing Flux Surfaces in R-Z Space (EFIT vs CHEASE ASCII/BINARY) ---")

function get_rz_grid(eq_obj)
    # 1. Define evaluation points
    # Use collect to ensure they are passed as arrays to bicube_eval
    # Use high density for theta to get smooth curves
    psi_g = collect(range(eq_obj.config.control.psilow, eq_obj.config.control.psihigh, length=20))
    theta_g = collect(range(0.0, 1.0, length=1000))
    
    # 2. Evaluate bicubic spline
    # fs is a (Npsi, Ntheta, 2) array
    fs = GeneralizedPerturbedEquilibrium.Spl.bicube_eval(eq_obj.rzphi, psi_g, theta_g)
    
    # 3. Extract components (using explicit indexing for clarity)
    # fs[:, :, 1] is rfac^2
    # fs[:, :, 2] is phase/2π
    rfac_sq = fs[:, :, 1]
    phase = fs[:, :, 2]
    
    # 4. Transformation logic (Poloidal -> Cartesian)
    # We use broadcasting (.) to handle the (Npsi, 1) + (1, Ntheta) + (Npsi, Ntheta) operations
    rfac = sqrt.(max.(0.0, rfac_sq))
    
    # eta = 2π * (theta + phase)
    # Note: theta_g' is a row vector, phase is a matrix. 
    # Julia's broadcasting handles this correctly: Matrix + RowVector
    eta = 2.0 * pi .* (theta_g' .+ phase)
    
    # Calculate R and Z
    R = eq_obj.ro .+ rfac .* cos.(eta)
    Z = eq_obj.zo .+ rfac .* sin.(eta)
    
    return R, Z
end

# Re-calculate with the fixed function
R_efit,   Z_efit   = get_rz_grid(plasma_eq_efit)
R_ascii,  Z_ascii  = get_rz_grid(plasma_eq_chease_ascii)  
R_binary, Z_binary = get_rz_grid(plasma_eq_chease_binary)  

# --- Plotting ---
p_surf = plot(title="Boundary & Flux Surface Comparison", 
              aspect_ratio=:equal, xlabel="R [m]", ylabel="Z [m]", size=(800, 900))

# Select indices: 1 (inner) to 20 (boundary)
surface_indices = [1, 5, 10, 15, 20]

for i in surface_indices
    # Set labels only for the boundary to keep the legend clean
    l_e = (i == 20) ? "EFIT" : ""
    l_a = (i == 20) ? "CHEASE ASCII" : ""
    l_b = (i == 20) ? "CHEASE Binary" : ""

    # Plot each surface. [val; val[1]] syntax closes the loop (connects end to start)
    plot!(p_surf, [R_efit[i, :]; R_efit[i, 1]], [Z_efit[i, :]; Z_efit[i, 1]], 
          color=:blue, lw=1.2, label=l_e)
    
    plot!(p_surf, [R_ascii[i, :]; R_ascii[i, 1]], [Z_ascii[i, :]; Z_ascii[i, 1]], 
          color=:red, ls=:dash, lw=1.2, label=l_a)
    
    plot!(p_surf, [R_binary[i, :]; R_binary[i, 1]], [Z_binary[i, :]; Z_binary[i, 1]], 
          color=:green, ls=:dot, lw=1.2, label=l_b)
end

display(p_surf)

## Debugging the difference

In [ ]:
println("\n=== DEBUGGING: Why are boundaries different? ===\n")

# Test at a specific psi value
test_psi = 0.5
test_theta = range(0.0, 1.0, length=100)

println("Testing at psi = $test_psi")
println("="^60)

# 1. Evaluate raw bicubic spline outputs
fs_efit = GeneralizedPerturbedEquilibrium.Spl.bicube_eval(plasma_eq_efit.rzphi, [test_psi], collect(test_theta))
fs_ascii = GeneralizedPerturbedEquilibrium.Spl.bicube_eval(plasma_eq_chease_ascii.rzphi, [test_psi], collect(test_theta))
fs_binary = GeneralizedPerturbedEquilibrium.Spl.bicube_eval(plasma_eq_chease_binary.rzphi, [test_psi], collect(test_theta))

println("\n1. Raw Spline Outputs (fs[:,:,1] = rfac², fs[:,:,2] = phase/2π)")
println("-"^60)
println("EFIT:")
println("  fs[:,:,1] range: [$(minimum(fs_efit[1,:,1])), $(maximum(fs_efit[1,:,1]))]")
println("  fs[:,:,2] range: [$(minimum(fs_efit[1,:,2])), $(maximum(fs_efit[1,:,2]))]")
println("\nCHEASE ASCII:")
println("  fs[:,:,1] range: [$(minimum(fs_ascii[1,:,1])), $(maximum(fs_ascii[1,:,1]))]")
println("  fs[:,:,2] range: [$(minimum(fs_ascii[1,:,2])), $(maximum(fs_ascii[1,:,2]))]")
println("\nCHEASE Binary:")
println("  fs[:,:,1] range: [$(minimum(fs_binary[1,:,1])), $(maximum(fs_binary[1,:,1]))]")
println("  fs[:,:,2] range: [$(minimum(fs_binary[1,:,1])), $(maximum(fs_binary[1,:,2]))]")

# 2. Compute rfac
rfac_efit = sqrt.(max.(0.0, fs_efit[1, :, 1]))
rfac_ascii = sqrt.(max.(0.0, fs_ascii[1, :, 1]))
rfac_binary = sqrt.(max.(0.0, fs_binary[1, :, 1]))

println("\n2. After sqrt (rfac)")
println("-"^60)
println("EFIT rfac:   [$(minimum(rfac_efit)), $(maximum(rfac_efit))]")
println("ASCII rfac:  [$(minimum(rfac_ascii)), $(maximum(rfac_ascii))]")
println("Binary rfac: [$(minimum(rfac_binary)), $(maximum(rfac_binary))]")

# 3. Compute eta
eta_efit = 2.0 * pi .* (test_theta .+ fs_efit[1, :, 2])
eta_ascii = 2.0 * pi .* (test_theta .+ fs_ascii[1, :, 2])
eta_binary = 2.0 * pi .* (test_theta .+ fs_binary[1, :, 2])

println("\n3. After eta = 2π*(θ + phase)")
println("-"^60)
println("EFIT eta:   [$(minimum(eta_efit)), $(maximum(eta_efit))]")
println("ASCII eta:  [$(minimum(eta_ascii)), $(maximum(eta_ascii))]")
println("Binary eta: [$(minimum(eta_binary)), $(maximum(eta_binary))]")

# 4. Magnetic axis positions
println("\n4. Magnetic Axis (R₀, Z₀)")
println("-"^60)
println("EFIT:   R₀ = $(plasma_eq_efit.ro),   Z₀ = $(plasma_eq_efit.zo)")
println("ASCII:  R₀ = $(plasma_eq_chease_ascii.ro),  Z₀ = $(plasma_eq_chease_ascii.zo)")
println("Binary: R₀ = $(plasma_eq_chease_binary.ro), Z₀ = $(plasma_eq_chease_binary.zo)")

# 5. Final R, Z
R_efit_test = plasma_eq_efit.ro .+ rfac_efit .* cos.(eta_efit)
Z_efit_test = plasma_eq_efit.zo .+ rfac_efit .* sin.(eta_efit)

R_ascii_test = plasma_eq_chease_ascii.ro .+ rfac_ascii .* cos.(eta_ascii)
Z_ascii_test = plasma_eq_chease_ascii.zo .+ rfac_ascii .* sin.(eta_ascii)

R_binary_test = plasma_eq_chease_binary.ro .+ rfac_binary .* cos.(eta_binary)
Z_binary_test = plasma_eq_chease_binary.zo .+ rfac_binary .* sin.(eta_binary)

println("\n5. Final R, Z coordinates")
println("-"^60)
println("EFIT:   R=[$(minimum(R_efit_test)), $(maximum(R_efit_test))], Z=[$(minimum(Z_efit_test)), $(maximum(Z_efit_test))]")
println("ASCII:  R=[$(minimum(R_ascii_test)), $(maximum(R_ascii_test))], Z=[$(minimum(Z_ascii_test)), $(maximum(Z_ascii_test))]")
println("Binary: R=[$(minimum(R_binary_test)), $(maximum(R_binary_test))], Z=[$(minimum(Z_binary_test)), $(maximum(Z_binary_test))]")

# 6. Plot step-by-step comparison
p1 = plot(test_theta, fs_efit[1, :, 1], label="EFIT", lw=2, title="Step 1: fs[:,:,1] (rfac²)")
plot!(p1, test_theta, fs_ascii[1, :, 1], label="ASCII", lw=2, ls=:dash)
plot!(p1, test_theta, fs_binary[1, :, 1], label="Binary", lw=2, ls=:dot)
xlabel!(p1, "θ (normalized)")

p2 = plot(test_theta, fs_efit[1, :, 2], label="EFIT", lw=2, title="Step 2: fs[:,:,2] (phase/2π)")
plot!(p2, test_theta, fs_ascii[1, :, 2], label="ASCII", lw=2, ls=:dash)
plot!(p2, test_theta, fs_binary[1, :, 2], label="Binary", lw=2, ls=:dot)
xlabel!(p2, "θ (normalized)")

p3 = plot(test_theta, rfac_efit, label="EFIT", lw=2, title="Step 3: rfac")
plot!(p3, test_theta, rfac_ascii, label="ASCII", lw=2, ls=:dash)
plot!(p3, test_theta, rfac_binary, label="Binary", lw=2, ls=:dot)
xlabel!(p3, "θ (normalized)")

p4 = plot(test_theta, eta_efit, label="EFIT", lw=2, title="Step 4: eta")
plot!(p4, test_theta, eta_ascii, label="ASCII", lw=2, ls=:dash)
plot!(p4, test_theta, eta_binary, label="Binary", lw=2, ls=:dot)
xlabel!(p4, "θ (normalized)")

p5 = plot(R_efit_test, Z_efit_test, label="EFIT", lw=2, title="Step 5: Final (R,Z)", aspect_ratio=:equal)
plot!(p5, R_ascii_test, Z_ascii_test, label="ASCII", lw=2, ls=:dash)
plot!(p5, R_binary_test, Z_binary_test, label="Binary", lw=2, ls=:dot)
xlabel!(p5, "R [m]")
ylabel!(p5, "Z [m]")

p_debug = plot(p1, p2, p3, p4, p5, layout=(2,3), size=(1500, 800))
display(p_debug)

# 7. Compute pairwise differences
println("\n6. Pairwise Differences")
println("-"^60)
println("fs[:,:,1] differences:")
println("  |ASCII - Binary|: max = $(maximum(abs.(fs_ascii[1,:,1] .- fs_binary[1,:,1])))")
println("  |EFIT - ASCII|:  max = $(maximum(abs.(fs_efit[1,:,1] .- fs_ascii[1,:,1])))")
println("  |EFIT - Binary|: max = $(maximum(abs.(fs_efit[1,:,1] .- fs_binary[1,:,1])))")

println("\nfs[:,:,2] differences:")
println("  |ASCII - Binary|: max = $(maximum(abs.(fs_ascii[1,:,2] .- fs_binary[1,:,2])))")
println("  |EFIT - ASCII|:  max = $(maximum(abs.(fs_efit[1,:,2] .- fs_ascii[1,:,2])))")
println("  |EFIT - Binary|: max = $(maximum(abs.(fs_efit[1,:,2] .- fs_binary[1,:,2])))")

println("\nMagnetic axis differences:")
println("  |ASCII - Binary| R₀: $(abs(plasma_eq_chease_ascii.ro - plasma_eq_chease_binary.ro))")
println("  |ASCII - Binary| Z₀: $(abs(plasma_eq_chease_ascii.zo - plasma_eq_chease_binary.zo))")
println("  |EFIT - ASCII| R₀:   $(abs(plasma_eq_efit.ro - plasma_eq_chease_ascii.ro))")
println("  |EFIT - ASCII| Z₀:   $(abs(plasma_eq_efit.zo - plasma_eq_chease_ascii.zo))")

println("\n" * "="^60)
println("Analysis complete!")